In [ ]:
import sys
import os
sys.path.append(os.path.realpath('./src/'))
import numpy as np
import torch
import matplotlib.pyplot as plt
import time
# torch.set_printoptions(precision=20,linewidth=100000000000000)
np.set_printoptions(precision=4,suppress=True,linewidth=100000000000000)
# sys.stdout = open('/dev/stdout', 'w')


Load the trained VAE here

checkpoint = {
            "model_state": self.vaeNet.state_dict(),
            "input_data": self.input_data,
            "scaling_info": self.scaling_info,
            "identifiers": self.identifiers,
            "architecture_config": self.architecture_config,
        }

In [ ]:
from vaeNet import VariationalAutoencoderModel
add_thickness = False
path = './savedNet/splineNet50k_n_cp_5_2x512nfeatures_14.pth'
if add_thickness:
    path = './savedNet/splineNet40k_n_cp_5_2x512nfeatures_15_zdim_2.pth'
checkpoint = torch.load(path,map_location='cpu',weights_only=False)
splineNet = VariationalAutoencoderModel(checkpoint['input_data'],checkpoint['scaling_info'],checkpoint['identifiers'],checkpoint['architecture_config'],useCPU=True)
splineNet.load_model_from_file(checkpoint['model_state'])
z = torch.tensor([[0,0],[0,1.0],[1.0,0],[1.0,1.0]]).to(dtype=splineNet.target_dtype)
z = torch.tensor([[-1.5,1.6]]).to(dtype=splineNet.target_dtype)
decodeprop = splineNet.vaeNet.predict(z)
decoded_prop_denormalized = splineNet.getProperties(decodeprop) # from last its Izz, Iyy, J, A, S, O, 
figLatent,axsLatent = splineNet.plot_latent_scatter(save_file='decoded_properties',\
                        save_figures=False,fig=plt.figure(figsize=(8,5)))
ax_box = None
print(checkpoint['scaling_info'])

cross-section plot and real properties calculator

In [ ]:
from splineGeom import SplineGeometry
from dataManager import DataManager
from geomProp import GeomProperties
# Design space parameters
n_cp = 5
k = 2

splineGeometry = SplineGeometry(n_cp, k=k, npt=1000)
geomPropCal = GeomProperties(splineGeometry)
dataManagerAgent = DataManager(geomPropCal)
CP_xy = decoded_prop_denormalized[0,0:-6].detach().numpy()
# CP_xy[-1] = 0.8
prop = geomPropCal.evaluate_section_metrics(CP_xy)
# clearer prints for decoded vs real properties
decoded_vals = decoded_prop_denormalized[0, -6:].detach().cpu().numpy()
real_vals = prop[1:]  # skip prop[0] as before

print(f"Decoded (denormalized) properties from VAE (indices {(n_cp-1)*2} ...):\n{decoded_vals}")
print(f"\nComputed real section metrics (full prop array):\n{prop}")
print(f"\nComparing decoded values to computed (prop[1:]):")

for i, (d, r) in enumerate(zip(decoded_vals, real_vals), start=1):
    print(f"  Property {i:>2}: decoded = {d:.6g}    computed = {r:.6g}    diff = {d - r:.6g}")
_,_=splineGeometry.plot_cross_section(CP_xy,prop)


In [ ]:
# find the bound for latent space points corresponding to valid cross-sections
data_idx = (checkpoint['input_data'][:,-6]==1)
data_ = checkpoint['input_data'][data_idx,:]            
valid_latent_points = splineNet.vaeNet.encode(data_.float()).cpu().detach().numpy()
min_bounds = np.min(valid_latent_points,axis=0)
max_bounds = np.max(valid_latent_points,axis=0)
print(f"Latent space bounds for valid cross-sections:\nMin: {min_bounds}\nMax: {max_bounds}")
min_set = np.array([0,-2.5])
max_set = np.array([2.5,2.5])
# find the 10% smaller bounds of min and max bounds without using minset and maxset
pct = 0.05
smaller_min_bounds = min_bounds + pct * (max_bounds - min_bounds)
smaller_max_bounds = max_bounds - pct * (max_bounds - min_bounds)
print(f"{pct*100}% smaller latent space bounds:\nMin: {smaller_min_bounds}\nMax: {smaller_max_bounds}")


In [ ]:
# sample_num = 1
# list_t = [0.0909,0.3636,0.727272,1.0] 
# # np.random.seed(1000)#1000 has pretty loop to test
# for i in list_t:
#     XY_all = np.array([[1.0,    1.0,    0.732, 0.268, 0.268, 0.732, 1.0,    1.0,    0.0909  ]])
#     XY_all[0,-1] = i    
#     XY_all = (torch.from_numpy(XY_all)* (checkpoint['scaling_info'][0, 0:XY_all.shape[1]] - checkpoint['scaling_info'][1, 0:XY_all.shape[1]]) + 
#                                     checkpoint['scaling_info'][1, 0:XY_all.shape[1]]).detach().numpy()
#     print(XY_all)
#     print(checkpoint['scaling_info'][0:2, 0:XY_all.shape[1]])
#     prop = geomPropCal.evaluate_section_metrics(XY_all.reshape(-1))
#     splineGeometry.plot_cross_section(XY_all.reshape(-1), prop,addLegend=False,addFill=True)
#     plt.savefig(f'PaperImg/cross_section_t_{int(np.round(XY_all[0,-1],decimals=1)*10)}by10.png',dpi=300)
#     # plt.show(block=True)

In [ ]:
def select_data_plot(pick_data_n = 50, Outcome=0):
    data_idx = (checkpoint['input_data'][:,-6]==Outcome)
    data_ = checkpoint['input_data'][data_idx,:]
    data_denormalized = splineNet.getProperties(data_[pick_data_n,:]).cpu().detach().numpy()
    CP_xy = data_denormalized[0:-6]
    prop  = data_denormalized[-6:]
    splineGeometry.plot_cross_section(CP_xy,prop,addLegend=True,addTitle=False,showSymm=True,annotateCP=True)

# select_data_plot(pick_data_n=550,Outcome=0)
# select_data_plot(pick_data_n=50,Outcome=1)
# select_data_plot(pick_data_n=1,Outcome=1) # circle
# select_data_plot(pick_data_n=13000,Outcome=1) # 


In [ ]:
# # plot the radial update of geometry
# rng = np.random.default_rng(0)
# theta = np.linspace(0,0.5*np.pi,5)
# r_rnd = rng.uniform(low=0.5,high=1.5,size=5)
# r_rnd = rng.uniform(low=0.5,high=1.5,size=5)
# r_rnd = 1.5*r_rnd/max(r_rnd)
# x_r1 = r_rnd*np.cos(theta)
# y_r1 = r_rnd*np.sin(theta)
# CP_xy_rnd1 = np.concatenate((x_r1[:-1],y_r1[1:]))
# prop = geomPropCal.evaluate_section_metrics(CP_xy_rnd1)
# ax_rad,_ = splineGeometry.plot_cross_section(CP_xy_rnd1,prop,addLegend=False,addTitle=False,showSymm=False,annotateCP=True)
# # draw line from origion to the control points
# for i in range(len(x_r1)):
#     ax_rad.plot([0,x_r1[i]],[0,y_r1[i]],'k--',linewidth=2.0)

# r_rnd = rng.uniform(low=0.5,high=1.5,size=5)
# r_rnd = .8*r_rnd/max(r_rnd)
# x_r = r_rnd*np.cos(theta)
# y_r = r_rnd*np.sin(theta)
# CP_xy_rnd = np.concatenate((x_r[:-1],y_r[1:]))
# ax_rad2,_ = splineGeometry.plot_cross_section(CP_xy_rnd,prop,ax=ax_rad,addLegend=False,addTitle=False,showSymm=False,annotateCP=False)


In [ ]:
def map_to_rectangle(z, h=1.0, k=0.0, width=2.0, height=5.0, theta=torch.tensor([0.0])):
    """
    Maps two unconstrained latent variables into a specified rectangular region.
    
    Args:
        z (torch.Tensor): Tensor of shape (N, 2). z[:, 0] and z[:, 1] are unconstrained. in 0 to 1 range
        h (float): x-coordinate of rectangle center.
        k (float): y-coordinate of rectangle center.
        width (float): Width of the rectangle.
        height (float): Height of the rectangle.
        theta (torch.Tensor): Rotation angle of the rectangle in radians.
    
    Returns:
        torch.Tensor: Mapped coordinates of shape (N, 2) inside the rotated rectangle.
    """
    
    if add_thickness == True:
        h=0.0; k=1.25; width=5.0; height=2.5; theta=torch.tensor([0.0])
    else:
        h=1.; k=0.0; width=2.0; height=5.0; theta=torch.tensor([0.0])
    
     # 1. Center to [-0.5, 0.5]
    z_centered = z - 0.5

    # 2. Scale to actual rectangle dimensions
    x_unrotated = z_centered[:, 0] * width
    y_unrotated = z_centered[:, 1] * height

    # 3. Rotate
    cos_theta = torch.cos(theta)
    sin_theta = torch.sin(theta)

    x_rotated = x_unrotated * cos_theta - y_unrotated * sin_theta
    y_rotated = x_unrotated * sin_theta + y_unrotated * cos_theta

    # 4. Translate to center (h, k)
    x_final = h + x_rotated
    y_final = k + y_rotated
    
    
    z_out = torch.stack((x_final, y_final), dim=1)
              

    return z_out
    
def plot_x_points_inlatentspace(x,fig,axs,lbls=None,boundOnly=False):
    
    if fig is None or axs is None:
        fig, axs = plt.subplots(figsize=(8,8))
    
    # only for helping with the plots of invalid shapes
    z_opt = torch.zeros_like(x)
    for i in range(x.shape[0]):
        if torch.any(x[i,:] < 0):
            z_opt[i,:] = -map_to_rectangle(torch.abs(x[i,:]).unsqueeze(0))
        else:
            z_opt[i,:] = map_to_rectangle(x[i,:].unsqueeze(0))
    
    # Safely remove previous optimization scatter if it exists (ax_opt may be a PathCollection or a list)
    try:
        if 'ax_opt' in globals() and ax_opt is not None:
            if isinstance(ax_opt, (list, tuple)):
                for art in ax_opt:
                    try:
                        art.remove()
                    except Exception:
                        pass
            else:
                try:
                    ax_opt.remove()
                except Exception:
                    pass
    except NameError:
        pass
    # Remove previous annotation texts if present
    try:
        if 'ax_opt_texts' in globals() and ax_opt_texts is not None:
            if isinstance(ax_opt_texts, (list, tuple)):
                for t in ax_opt_texts:
                    try:
                        t.remove()
                    except Exception:
                        pass
            else:
                try:
                    ax_opt_texts.remove()
                except Exception:
                    pass
    except NameError:
        pass
        
    if boundOnly:
        # Plot rectangle boundary using 4 corner points
        corner_pt = torch.tensor([[0, 0], [1, 0], [1, 1], [0, 1]])
        corner_mapped = map_to_rectangle(corner_pt)
        corner_mapped = torch.cat((corner_mapped, corner_mapped[0, :].unsqueeze(0)), dim=0)  # Close the rectangle loop
        ax_opt, = axs.plot(corner_mapped[:, 0].detach().numpy(), corner_mapped[:, 1].detach().numpy(), color='black',linestyle='--', linewidth=2, label=lbls)
        
        
    else:
        # Plot optimized points and annotate each with its latent index
        ax_opt = axs.scatter(z_opt[:, 0].detach().numpy(), z_opt[:, 1].detach().numpy(), color='blue', s=100, marker='o', label=lbls)
        
        # Create new annotations and store the Text artist objects in ax_opt_texts
        ax_opt_texts = []
        for i_pt in range(z_opt.shape[0]):
            x_pt = z_opt[i_pt, 0].item()
            y_pt = z_opt[i_pt, 1].item()
            txt = axs.text(x_pt+0.05, y_pt+0.05, f'$z_{i_pt}$', fontsize=14, color='black', fontweight='heavy')
            ax_opt_texts.append(txt)
        
        
    # axs.legend(loc='best')
    return fig,axs
    


In [ ]:
def propPredict(x_in,plot_invalid_shapes=False):
    """
    x_in: torch.Tensor or array-like with shape (N , n), values expected in [0,1].
    Returns: torch.Tensor of shape (N,5) with properties [Sf, A, J, Iy, Iz] (last 5 properties).
    """
    

    z_in = x_in[:,0:2] # N,latentDim
    k = x_in[:,2] + 0.001 # scale factor
    theta = x_in[:,3]* torch.pi/2
    
    if plot_invalid_shapes:
         # only for helping with the plots of invalid shapes
        z_latent = torch.zeros_like(z_in)
        for i in range(z_in.shape[0]):
            if torch.any(z_in[i,:] < 0):
                z_latent[i,:] = -map_to_rectangle(torch.abs(z_in[i,:]).unsqueeze(0).to(dtype=splineNet.target_dtype))
            else:
                z_latent[i,:] = map_to_rectangle(z_in[i,:].unsqueeze(0).to(dtype=splineNet.target_dtype))
    else:
        z_latent = map_to_rectangle(z_in.to(dtype=splineNet.target_dtype)) #type: ignore
    
    
    decodeprop = splineNet.vaeNet.predict(z_latent)
    decoded_prop_denormalized = splineNet.getProperties(decodeprop)
    SAJI = decoded_prop_denormalized[:, -5:]  # Sf, A, J, Iy, Iz (last 5)
    
    Sf = SAJI[:, 0]
    A = SAJI[:, 1]
    J = SAJI[:, 2]
    Iy = SAJI[:, 3]
    Iz = SAJI[:, 4]
    
    # Apply scaling
    A_dash = A * k**2
    J_dash = J * k**4
    Iy_dash = Iy * k**4
    Iz_dash = Iz * k**4
    
    # Apply rotation
    cos_t = torch.cos(theta)
    sin_t = torch.sin(theta)
    Iy_final = Iy_dash * cos_t**2 + Iz_dash * sin_t**2
    Iz_final = Iy_dash * sin_t**2 + Iz_dash * cos_t**2
    
    SAJI_new = torch.stack([Sf, A_dash, J_dash, Iy_final, Iz_final], dim=1)
    
    return SAJI_new.to(dtype=x_in.dtype)

def shapePredict(x_in,plot_invalid_shapes=False):
    """
    x_in: torch.Tensor or array-like with shape (N , n), values expected in [0,1].
    Returns: torch.Tensor of shape (N,number_of_control_points) with control point coordinates.
    """
        
    z_in = x_in[:,0:2] # N,latentDim
    k = x_in[:,2] # scale factor
    theta = x_in[:,3]* torch.pi/2
    
    
    if plot_invalid_shapes:
         # only for helping with the plots of invalid shapes
        z_latent = torch.zeros_like(z_in)
        for i in range(z_in.shape[0]):
            if torch.any(z_in[i,:] < 0):
                z_latent[i,:] = -map_to_rectangle(torch.abs(z_in[i,:]).unsqueeze(0).to(dtype=splineNet.target_dtype))
            else:
                z_latent[i,:] = map_to_rectangle(z_in[i,:].unsqueeze(0).to(dtype=splineNet.target_dtype))
    else:
        z_latent = map_to_rectangle(z_in.to(dtype=splineNet.target_dtype)) #type: ignore
    
    # z_latent = map_to_rectangle(z_in.to(dtype=splineNet.target_dtype)) #type: ignore
    decodeprop = splineNet.vaeNet.predict(z_latent)
    decoded_prop_denormalized = splineNet.getProperties(decodeprop)
    CP_xy = decoded_prop_denormalized[:, 0:-6]  # Sf, A, J, Iy, Iz (last 5)
    CP_xy = CP_xy * k.unsqueeze(1)
    return CP_xy.to(dtype=x_in.dtype), theta



This cell mainly for visulaization

In [ ]:
# z_all_init = torch.tensor([[0.0,0.0],[0,1],[1,0],[1.0,1.0]])
# n=4
# # Spiral parameters
# theta = np.linspace(0, 2*np.pi, n)  # angle
# a = 0.0                                 # starting radius
# b = 0.05                                # how tightly the spiral grows
# r = a + b * theta                       # radius as function of angle
# # Convert to x,y and shift to start at (0.5, 0.5)
# x = r * np.cos(theta) + 0.5
# y = r * np.sin(theta) + 0.5

# z_valid = np.concatenate((x.reshape((-1,1)),y.reshape((-1,1))),axis=1)
# z_valid = torch.tensor(z_valid,dtype=torch.float32)
# z_invalid = z_valid.clone()
# z_invalid[:,1] = z_valid[:,1] - 1
# z_all_init = torch.cat((z_valid,z_invalid),dim=0)

# z_all = torch.cat((z_all_init, torch.ones_like(z_all_init)), axis=1)
# CP_xy, theta_all = shapePredict(z_all,plot_invalid_shapes=True)
# prop = propPredict(z_all,plot_invalid_shapes=True)
# prop = torch.cat((torch.ones_like(prop[:,0]).reshape((-1,1)),prop),dim=1)
# # XY_all = []
# for i in range(CP_xy.shape[0]):
#     a, _ = splineGeometry.plot_cross_section(CP_xy[i,:].detach().numpy(),prop[i,:].detach().numpy(),theta=theta_all[i].item(),addLegend=False, addTitle=False)
#     # XY_all.append(xy_final[np.newaxis,:,:])
    
# f_L, _ = plot_x_points_inlatentspace(z_all_init, figLatent, axsLatent, lbls = 'z')
# f_L




In [ ]:

# #####################################################################################################
# create z_grid for counter plotting
z1 = torch.linspace(0,1,50)
z2 = torch.linspace(0,1,50)
Z1, Z2 = torch.meshgrid(z1, z2, indexing='ij')
Z_grid = torch.stack((Z1.reshape(-1),Z2.reshape(-1)),dim=1)
z_latent = map_to_rectangle(Z_grid.to(dtype=splineNet.target_dtype)) #type: ignore
Z_grid = torch.cat((Z_grid, torch.ones_like(Z_grid)), axis=1);Z_grid[:,-1] = 0.0; # type: ignore
prop_grid = propPredict(Z_grid,plot_invalid_shapes=False).detach()
Sf_grid = prop_grid[:,0].reshape(Z1.shape)
A_grid = prop_grid[:,1].reshape(Z1.shape)
J_grid = prop_grid[:,2].reshape(Z1.shape)
Iy_grid = prop_grid[:,3].reshape(Z1.shape)
Iz_grid = prop_grid[:,4].reshape(Z1.shape)
Z1 = z_latent[:,0].reshape(Z1.shape)
Z2 = z_latent[:,1].reshape(Z1.shape)

sizen=5; scale = sizen / 5.0
fontXlabel = 18*scale; fontTick = 16*scale; fontClabel = 10*scale; lvls = 20

fig_contour1, axs_contour1 = plt.subplots(1,3, figsize=(3*sizen,sizen))
cs1 = axs_contour1[0].contour(Z1.numpy(), Z2.numpy(), Sf_grid.numpy(), levels=lvls, cmap='jet')
cbar = fig_contour1.colorbar(cs1, ax=axs_contour1[0])
cbar.ax.tick_params(labelsize=fontTick)  
axs_contour1[0].clabel(cs1, inline=True, fontsize=fontClabel)
axs_contour1[0].set_title('Contour of S',fontsize=fontXlabel)

cs2 = axs_contour1[1].contour(Z1.numpy(), Z2.numpy(), A_grid.numpy(), levels=lvls, cmap='jet')
cbar = fig_contour1.colorbar(cs2, ax=axs_contour1[1])
cbar.ax.tick_params(labelsize=fontTick)  
axs_contour1[1].clabel(cs2, inline=True, fontsize=fontClabel)
axs_contour1[1].set_title('Contour of A',fontsize=fontXlabel)

cs3 = axs_contour1[2].contour(Z1.numpy(), Z2.numpy(), J_grid.numpy(), levels=lvls, cmap='jet')
cbar = fig_contour1.colorbar(cs3, ax=axs_contour1[2])
cbar.ax.tick_params(labelsize=fontTick)  
axs_contour1[2].clabel(cs3, inline=True, fontsize=fontClabel)
axs_contour1[2].set_title('Contour of J',fontsize=fontXlabel)

fig_contour2, axs_contour2 = plt.subplots(1,2, figsize=(2*sizen*0.83,sizen*0.92))
cs4 = axs_contour2[0].contour(Z1.numpy(), Z2.numpy(), Iy_grid.numpy(), levels=lvls, cmap='jet')
cbar = fig_contour2.colorbar(cs4, ax=axs_contour2[0])
cbar.ax.tick_params(labelsize=fontTick)  
axs_contour2[0].clabel(cs4, inline=True, fontsize=fontClabel)
axs_contour2[0].set_title('Contour of Ixx',fontsize=fontXlabel)

cs5 = axs_contour2[1].contour(Z1.numpy(), Z2.numpy(), Iz_grid.numpy(), levels=lvls, cmap='jet')
cbar = fig_contour2.colorbar(cs5, ax=axs_contour2[1])
cbar.ax.tick_params(labelsize=fontTick)  
axs_contour2[1].clabel(cs5, inline=True, fontsize=fontClabel)
axs_contour2[1].set_title('Contour of Iyy',fontsize=fontXlabel)

plt.tight_layout()
# add Latent Dim 1 and Latent Dim 2 are x and y axis labels
axs_contour1[0].set_ylabel('Latent Dim 2', fontsize=fontXlabel)
for ax in axs_contour1.flat:
    ax.set_xlabel('Latent Dim 1', fontsize=fontXlabel)
    ax.tick_params(axis='both', which='major', labelsize=fontTick)
    

axs_contour2[0].set_ylabel('Latent Dim 2', fontsize=fontXlabel)
for ax in axs_contour2.flat:
    ax.set_xlabel('Latent Dim 1', fontsize=fontXlabel)
    ax.tick_params(axis='both', which='major', labelsize=fontTick)

# save fig axs_contour1 and axs_contour2
# Save the figures
fig1 = axs_contour1[0].figure  # get the figure object from axes
fig2 = axs_contour2[0].figure

fig1.savefig('PaperImg/LatentSpaceContour1.png', dpi=300, bbox_inches='tight')
fig2.savefig('PaperImg/LatentSpaceContour2.png', dpi=300, bbox_inches='tight')



In [ ]:
# # plot only the optimization boundary in latent space
# f_L, _ = plot_x_points_inlatentspace(torch.zeros((4,2)), figLatent, axsLatent, lbls = 'z', boundOnly=True)
# f_L

In [ ]:
# # plot the effect of scaling and rotations
# z_base = torch.tensor([[0.6,0.6]])
# s = torch.linspace(0.2,1,3)
# t = torch.linspace(1,0,3)
# param_st_grid = torch.cartesian_prod(t, s)
# param_st_grid = torch.fliplr(param_st_grid)
# print(param_st_grid)
# z_base = z_base.repeat(param_st_grid.shape[0],1)
# x_all = torch.cat((z_base,param_st_grid),dim=1)
# CP_xy,theta_all = shapePredict(x_all)
# prop = propPredict(x_all).detach()
# prop = torch.cat((torch.ones_like(prop[:,0]).reshape((-1,1)),prop),dim=1)

# # Define grid dimensions
# n_rows, n_cols = 3, 3

# # fig, axes = plt.subplots(n_rows, n_cols, figsize=(15,15))
# fig = plt.figure(figsize=(15, 15))

# gs = fig.add_gridspec(
#     nrows=n_rows,
#     ncols=n_cols,
#     left=0.15,    # ← extra space on the left
#     bottom=0.1,  # ↓ extra space at the bottom
#     right=0.95,
#     top=0.95,
#     wspace=0.4,
#     hspace=0.4
# )
# axes_in_grid = [fig.add_subplot(gs[i, j]) for i in range(3) for j in range(3)]
# for i in range(CP_xy.shape[0]):
#     a, _ = splineGeometry.plot_cross_section(CP_xy[i,:].detach().numpy(),prop[i,:].detach().numpy(),ax=axes_in_grid[i],theta=theta_all[i].item(),addLegend=False, addTitle=True)

# # font sizes control
# fontsize_label = 24
# fontsize_ticks = 18
# #params for axis lines and labels
# x_loc_st = 0.1
# x_loc_ed = 1.
# y_loc_st = 0.075
# y_loc_ed = 1.
# lbl_x_loc = [0.57,0.02]
# lbl_y_loc = [0.02,0.525]

# # Add axis lines around the subplot grid
# # Bottom line
# fig.add_artist(plt.Line2D([x_loc_st, x_loc_ed], [y_loc_st, y_loc_st], transform=fig.transFigure, 
#                           color='black', linewidth=2.5))
# # Left line
# fig.add_artist(plt.Line2D([x_loc_st, x_loc_st], [y_loc_st, y_loc_ed], transform=fig.transFigure, 
#                           color='black', linewidth=2.5))

# # Add figure-level axis labels
# fig.text(lbl_x_loc[0], lbl_x_loc[1], 'Scaling (s)', ha='center', fontsize=fontsize_label)
# fig.text(lbl_y_loc[0], lbl_y_loc[1], 'Rotation angle (θ)', va='center', rotation='vertical', fontsize=fontsize_label)

# # Create tick labels for s and t
# s_ticks = [f'{v:.2f}' for v in s.numpy()]
# t_ticks = [f'{v:.2f}' for v in t.numpy()]
# t_ticks = [
#     r'$\mathbf{\frac{\pi}{2}}$', 
#     r'$\mathbf{\frac{\pi}{4}}$', 
#     r'$\mathbf{0}$'
# ]

# dash_len = 0.01

# # Add tick labels along bottom (s values)
# for j, label in enumerate(s_ticks):
#     x_loc,y_loc = 0.155+x_loc_st+ (j / n_cols) * 0.88,y_loc_st - 0.02
#     fig.text(x_loc, y_loc, label, ha='center', fontsize=fontsize_ticks, fontweight='bold')
#     fig.add_artist(plt.Line2D( [x_loc, x_loc],
#                                [y_loc+0.015, y_loc+0.015+dash_len],                                 
#                                   color='black', linestyle='-', linewidth=1.8))

# # Add tick labels along left side (t values - reversed for top to bottom)
# for i, label in enumerate(t_ticks):
#     x_loc,y_loc = x_loc_st-0.03, 0.21 + ((n_rows-1-i) / n_rows) * 0.935
#     if i<2:
#         fig.text(x_loc, y_loc, label, ha='center', fontsize=fontsize_ticks*1.5, fontweight='bold')
#     else:
#         fig.text(x_loc, y_loc, label, ha='center', fontsize=fontsize_ticks, fontweight='bold')    
#     fig.add_artist(plt.Line2D([x_loc+0.025, x_loc+0.025+dash_len],
#                                   [y_loc+0.005, y_loc+0.005],
#                                   color='black', linestyle='-', linewidth=1.8))

# plt.tight_layout(rect=[0.05, 0.05, 1, 1])

In [ ]:
# # find denormalized value for z
# z_in = torch.tensor([[0.94181615,0.78384819],
#                      [0.99744117,0.18707824],
#                      [0.99651434,0.1881962 ],
#                      [0.76775844,0.68977749]]).to(dtype=splineNet.target_dtype)
# z_in = torch.tensor([[0.3606, 0.9995]])

# z_latent = map_to_rectangle(z_in)
# print(z_latent)


In [ ]:
# np.set_printoptions(precision=4, suppress=True)
# x = torch.tensor([[0.3606, 0.9995,1.    ,0.3989]])
# CP_xy, theta_all = shapePredict(x)
# prop = propPredict(x)
# prop = torch.cat((torch.ones_like(prop[:,0]).reshape((-1,1)),prop),dim=1).detach().numpy()

# CP_xy= CP_xy.reshape((1,-1))
# XY_flat = CP_xy[:,:-1].reshape((-1,))
        
# xy_quarter, XY_q = splineGeometry.evaluate_bspline_contour(XY_flat.detach().numpy())
# xy_quarter_eroded = splineGeometry.erode_cross_section(CP_xy.detach().numpy().reshape((-1,)))


# coarseIndex = np.linspace(0, xy_quarter_eroded.shape[0] - 1, 10, dtype=int) # type: ignore
# xy_quarter_coarse_eroded = xy_quarter_eroded[coarseIndex, :]
# # print(xy_quarter_eroded,xy_quarter_coarse_eroded)
# # J, phi_sol, mesh = geomPropCal.solve_prandtl_quadrant(xy_quarter_coarse_eroded)
# # geomPropCal.plot_solution(mesh,phi_sol,J,fig=None)

# r_quater = np.sqrt(xy_quarter[:,0]**2 + xy_quarter[:,1]**2)
# r_eroded = np.sqrt(xy_quarter_eroded[:,0]**2 + xy_quarter_eroded[:,1]**2)
# print(f"Eroded min radius: {np.min(r_eroded):.4f}, Original min radius: {np.min(r_quater):.4f}")
# print(f"Eroded max radius: {np.max(r_eroded):.4f}, Original max radius: {np.max(r_quater):.4f}")
# print(f"Eroded mean radius: {np.mean(r_eroded):.4f}, Original mean radius: {np.mean(r_quater):.4f}")

# # prop_real = geomPropCal.evaluate_section_metrics(CP_xy.detach().numpy().reshape((-1,)))
# # print("hollow", prop_real)
# # prop_solid = geomPropCal.evaluate_section_metrics(XY_flat.detach().numpy())
# # print("solid", prop_solid)

# print("Area of original shape:" , np.pi*np.mean(r_quater)**2)
# print(f"Area of eroded shape: {np.pi*np.mean(r_eroded)**2:.4f}")
# print(f"Area of hollow by difference: {np.pi*np.mean(r_quater)**2 - np.pi*np.mean(r_eroded)**2:.4f}")

# fig_eroded, ax_eroded = plt.subplots(1,1, figsize=(12,12))
# plt.plot(xy_quarter[:,0],xy_quarter[:,1],'b-',label='Original shape')
# plt.plot(xy_quarter_eroded[:,0],xy_quarter_eroded[:,1],'r--',label='Eroded shape')
# plt.plot(xy_quarter_coarse_eroded[:,0],xy_quarter_coarse_eroded[:,1],'go--',label='Eroded shape coarse')
# for i in range(xy_quarter_coarse_eroded.shape[0]):
#     plt.text(xy_quarter_coarse_eroded[i,0]+0.02,xy_quarter_coarse_eroded[i,1]+0.02,f'{i}',fontsize=8,color='green')
# plt.axis('equal')
# plt.title('Eroded cross-sectional shape')
# plt.legend()
# plt.xlim([0,1.5])
# plt.show()
# print(aaa)

In [ ]:
# # plot the effect of z1 and z2


# # Define grid dimensions
# n_rows, n_cols = 6,6

# st_base = torch.tensor([[1.0,0.0]])

# z1 = torch.linspace(0.0,1,n_cols)
# z2 = torch.linspace(0.0,1,n_rows)
# param_grid = torch.cartesian_prod(z1, z2)# 0.05+
# st_base = st_base.repeat(param_grid.shape[0],1)
# x_all = torch.cat((param_grid,st_base),dim=1)
# CP_xy,theta_all = shapePredict(x_all)
# prop = propPredict(x_all).detach()
# prop = torch.cat((torch.ones_like(prop[:,0]).reshape((-1,1)),prop),dim=1)
# # fig, axes = plt.subplots(n_rows, n_cols, figsize=(15,15))
# fig = plt.figure(figsize=(15, 15))

# gs = fig.add_gridspec(
#     nrows=n_rows,
#     ncols=n_cols,
#     left=0.15,    # ← extra space on the left
#     bottom=0.1,  # ↓ extra space at the bottom
#     right=0.95,
#     top=0.95,
#     wspace=0.4,
#     hspace=0.4
# )
# axes_in_grid = [fig.add_subplot(gs[i, j]) for i in range(n_rows) for j in range(n_cols)]
# axes = np.array(axes_in_grid).reshape(n_rows, n_cols)
# axes_flipped = np.flipud(axes).T
# axes_in_grid = axes_flipped.flatten().tolist()
# for i in range(CP_xy.shape[0]):
#     # _, _ = splineGeometry.plot_cross_section(CP_xy[i,:].detach().numpy(),prop[i,:].detach().numpy(),theta=theta_all[i].item(),addLegend=False, addTitle=False)
#     a, _ = splineGeometry.plot_cross_section(CP_xy[i,:].detach().numpy(),prop[i,:].detach().numpy(),ax=axes_in_grid[i],theta=theta_all[i].item(),addLegend=False, addTitle=False)

# # font sizes control
# fontsize_label = 26
# fontsize_ticks = 18
# #params for axis lines and labels
# x_loc_st = 0.1
# x_loc_ed = 1.
# y_loc_st = 0.075
# y_loc_ed = 1.
# lbl_x_loc = [0.57,0.02]
# lbl_y_loc = [0.02,0.525]

# # Add axis lines around the subplot grid
# # Bottom line
# fig.add_artist(plt.Line2D([x_loc_st, x_loc_ed], [y_loc_st, y_loc_st], transform=fig.transFigure, 
#                           color='black', linewidth=2.5))
# # Left line
# fig.add_artist(plt.Line2D([x_loc_st, x_loc_st], [y_loc_st, y_loc_ed], transform=fig.transFigure, 
#                           color='black', linewidth=2.5))

# # Add figure-level axis labels
# fig.text(lbl_x_loc[0], lbl_x_loc[1], 'Latent Dim 1', ha='center', fontsize=fontsize_label)
# fig.text(lbl_y_loc[0], lbl_y_loc[1], 'Latent Dim 2', va='center', rotation='vertical', fontsize=fontsize_label)

# # Create tick labels for s and t
# z1_ticks = [f'{v:.2f}' for v in 2*z1.numpy()]
# z2_ticks = [f'{v:.2f}' for v in (np.flip(z2.numpy())*5 - 2.5)]
# if add_thickness:
#     # Create tick labels for s and t
#     z1_ticks = [f'{v:.2f}' for v in (-2.5 + 5*z1.numpy())]
#     z2_ticks = [f'{v:.2f}' for v in np.flip(z2.numpy())*2.]
    
# dash_len = 0.01

# # Add tick labels along bottom (s values)
# for j, label in enumerate(z1_ticks):
#     x_loc,y_loc = 0.1+x_loc_st+ (j / n_cols) * 0.84,y_loc_st - 0.02
#     fig.text(x_loc, y_loc, label, ha='center', fontsize=fontsize_ticks, fontweight='bold')
#     fig.add_artist(plt.Line2D( [x_loc, x_loc],
#                                [y_loc+0.015, y_loc+0.015+dash_len],                                 
#                                   color='black', linestyle='-', linewidth=1.8))

# # Add tick labels along left side (t values - reversed for top to bottom)
# for i, label in enumerate(z2_ticks):
#     x_loc,y_loc = x_loc_st-0.03, 0.149 + ((n_rows-1-i) / n_rows) * 0.89
#     # if i<2:
#     #     fig.text(x_loc, y_loc, label, ha='center', fontsize=fontsize_ticks*1.5, fontweight='bold')
#     # else:
#     fig.text(x_loc, y_loc, label, ha='center', fontsize=fontsize_ticks, fontweight='bold')    
#     fig.add_artist(plt.Line2D([x_loc+0.025, x_loc+0.025+dash_len],
#                                   [y_loc+0.005, y_loc+0.005],
#                                   color='black', linestyle='-', linewidth=1.8))

# plt.tight_layout(rect=[0.05, 0.05, 1, 1])
# # fig.savefig('PaperImg/LatentSpaceShapesAxis.png', dpi=300)

In [ ]:
from examples import getExample
from frameFE3 import FrameFE


params = {'base': 2.,'height':5.,'phi':0.0,'delta':0.0, 'nx': 3, 'ny':3, 'nz':1,
            'Name':'C4','Shape':'Square'}
scale = 25.4 # inch to mm 
H = 0.5*scale# 0.5 inch per row height
W = 1*scale  # 1 inch
B = 1*scale  # 1 inch
params['height'] =  H
params['base'] =  H
params['Name']='C4'

E = torch.tensor([17.0])
numElemsPerBeam = 4 #int(864/connectivity.shape[0]) # 864 for K, 1728 for C, numElemsPerBeam = 5 for
elemSize = 3.0 #0.8
matrixType = 'Sparse'# Sparse or Dense
Section = 'circle' # for plotting purposes here

# 3 for kelvin lattice, 2 is 3 beam problem, 1 for 1 beam, 
# 4 for the cantilever, 5 for MBB, 6 for compliant mech
exampleCase = '1fz' 

if int(exampleCase[0]) < 3:
    params = None
if int(exampleCase[0]) == 4 or int(exampleCase[0]) == 5:
    params['Name']='B'# B for body centered cubic, O for octet
    params['nx'] = 10
    params['ny'] = 3
    params['nz'] = 3
if int(exampleCase[0]) == 6:
    params['Name']='B'# B for body centered cubic, O for octet
    params['nx'] = 6    
    params['ny'] = 1
    params['nz'] = 4

    
exampleInfo, nodeXY, connectivity, bc, radiiNodIndex,radiiElemIndex = getExample(exampleCase,params) ##3D beam model
if int(exampleCase[0]) == 6:
    nodeXY[:,1] = nodeXY[:,1] * 0.5
numUnitLatElem = max(radiiElemIndex) + 1
numUnitLatNode = max(radiiNodIndex)  + 1 
meshSetting = {'nodeXY':nodeXY,'connectivity':connectivity,'numElemsPerBeam':numElemsPerBeam,'elemSize':elemSize,
               'ElemType':'TS','radiiElemIndex':radiiElemIndex,'radiiNodIndex':radiiNodIndex,
               'numUnitLatElem':numUnitLatElem,'numUnitLatNode':numUnitLatNode}# and TS - Timoshenko and EB - Euler-Bernoulli
# print(meshSetting)
    
solverName = 'scipy'# spsolve for sparse
solverName = 'pypardiso'# pypardiso for sparse
# solverName = 'cg'# conjugate gradient for sparse
AnalysisSettings = {'Type':'Linear','solver':solverName,'matrixType':matrixType,'Section':Section}# mode {analysis or optimization} true uses torch sp solve
frameFE = FrameFE(meshSetting,bc,AnalysisSettings)
# axFE = frameFE.plotStructure('Linear',plotDeformed = False,TrueScale=True,
#         fig=plt.figure(10,figsize=(20,20)),thicknessPlot=False,nodeAnnotate=False)
specialPlot =False
if specialPlot:
    # remove grid, title, label from the plot
    axFE.grid(False)
    axFE.set_title('')
    axFE.set_xlabel('')
    axFE.set_ylabel('')
    axFE.set_zlabel('')
    # remove axis ticks
    axFE.set_xticks([])
    axFE.set_yticks([])
    axFE.set_zticks([])
    # save figure
    plt.savefig('PaperImg/frame_structure.png', dpi=300)
    # add blue plane at X=0 and red plane at X= max X with 50% transparency
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    import numpy as np
    
    x_min, x_max = np.min(nodeXY[:, 0]), np.max(nodeXY[:, 0])
    y_min, y_max = np.min(nodeXY[:, 1]), np.max(nodeXY[:, 1])
    z_min, z_max = np.min(nodeXY[:, 2]), np.max(nodeXY[:, 2])
    
    # ---- Plane at X = x_min (blue, hatched) ----
    verts_xmin = [[
        (x_min, y_min, z_min),
        (x_min, y_max, z_min),
        (x_min, y_max, z_max),
        (x_min, y_min, z_max),
    ]]
    
    plane_xmin = Poly3DCollection(
        verts_xmin,
        facecolor='blue',
        alpha=0.5,
        hatch='/////',
        edgecolor='k'
    )
    axFE.add_collection3d(plane_xmin)
    
    # ---- Plane at X = x_max (red, hatched) ----
    verts_xmax = [[
        (x_max, y_min, z_min),
        (x_max, y_max, z_min),
        (x_max, y_max, z_max),
        (x_max, y_min, z_max),
    ]]
    
    plane_xmax = Poly3DCollection(
        verts_xmax,
        facecolor='red',
        alpha=0.8,
        hatch='\\\\',
        edgecolor='k'
    )
    axFE.add_collection3d(plane_xmax)
    plt.savefig('PaperImg/frame_structure_bc.png', dpi=300)

plt.show()

symType = 'Independent'  # 'OneValue', 'Midplane','Unitcell', 'Independent'
if int(exampleCase[0]) >= 3 and int(exampleCase[0]) < 6:
    symArray = None # enforces Unitcell symmetry for lattices

    if symType == 'Unitcell':
        symArray = np.arange(frameFE.meshSetting['numUnitLatElem']).tolist()  # one unitcell is unique
    elif symType == 'OneValue':
        symArray = [np.arange(numUnitLatElem)] # makes all the crossectins to be mapped to one design
    elif symType == 'Independent':
        symArray = np.arange(connectivity.shape[0]).tolist() # each crosssection independent
    elif symType == 'Midplane':
        lattice_midpoint = np.mean(nodeXY[0:numUnitLatElem, :], axis=0)
        midpoints = (nodeXY[connectivity[0:numUnitLatElem,0],:] + nodeXY[connectivity[0:numUnitLatElem,1],:]) / 2
        distances = np.round((midpoints[:,-1] - lattice_midpoint[-1])/np.linalg.norm(midpoints[:,-1] - lattice_midpoint[-1]), decimals=5)
        # this splits distances into 3 groups: positive, negative, zero
        distances =  (distances > 0.0)* 1.0 + (distances < 0.0)* 2.0 + (distances == 0.0)* 3.0
        
        unique_rows, indices = np.unique(distances, return_inverse=True)
        symArray = []
        for i in range(len(unique_rows)):
            symArray.append(np.where(indices == i)[0])
elif int(exampleCase[0]) == 6:
    # this is to apply side symmetry for the frame example
    nodeXYNormalized = nodeXY/np.max(np.abs(nodeXY),axis=0)
    midpoints = (nodeXYNormalized[connectivity[:,0],:] + nodeXYNormalized[connectivity[:,1],:]) / 2
    midpoints[:,0] = np.abs(midpoints[:,0] - 0.5) # ceneter it at 0.5
    midpoints = np.round(midpoints,decimals=4)
    # find unique set of values in midpoints along rows and get their indices
    unique_rows, indices = np.unique(midpoints, axis=0, return_inverse=True)
    symArray = []
    for i in range(len(unique_rows)):
        symArray.append(np.where(indices == i)[0])

else:
    symArray = None
    
print(np.min(nodeXY,axis=0), np.max(nodeXY,axis=0))

Optimization 

In [ ]:
from optimizeFrame3 import OptimizeFrame
   
Section = 'z' # circle or z

varType = 'Element' # capable: Element or Node # Node results are supressed in this work
rng = np.random.default_rng(4)#3 or 4-paper, or 44 for compliant mech
if Section == 'circle':
    if varType == 'Element':
        x0 = rng.uniform(size = (numUnitLatElem,1))
        if symType == 'Independent':
            x0 = rng.uniform(size = (connectivity.shape[0],1)) # base connectivity
    max_x = 1.5
    min_x = 1e-4
else:
    max_x = 1.0# latent space inhertly 1.5 max
    min_x = 0.0
    if varType == 'Element':
        x0 = rng.uniform(size = (numUnitLatElem,4)) # z0, z1, k, theta
        if symType == 'Independent':
            x0 = rng.uniform(size = (connectivity.shape[0],4)) # z0, z1, k, theta, # base connectivity

x_init = x0.reshape((-1,1))
varSetup = {'Section':Section,'min_x':min_x,'max_x':max_x,'symArray':symArray,'varType':varType,'propPredict':propPredict}
matProp = {'E':E}

optFrame = OptimizeFrame(frameFE,matProp,exampleInfo,varSetup)

start_time = time.time()
vc,vsen = optFrame.VolumeCon(x_init)
C,dCdr = optFrame.objectiveSE(x_init,Jac=True)
# C_mse,dCdr = optFrame.objectiveMSEbySE(x_init,Jac=True)

# mfc,mfsen = optFrame.ManufacturabilityCon(x_init)
# print(C)
# print("Elapsed time:", time.time() - start_time)
# optFrame.frameFE.plotStructure('Linear',plotDeformed = True,TrueScale=True,
#                 fig=plt.figure(10,figsize=(10,10)),thicknessPlot=False,nodeAnnotate=False)
# optFrame.frameFE.ax.view_init(elev=0., azim=-90)
# plt.show()
# optFrame.frameFE.plotStructure('Linear',plotDeformed = True,TrueScale=True,
#                 fig=plt.figure(10,figsize=(10,10)),thicknessPlot=False,nodeAnnotate=False)
# optFrame.frameFE.ax.view_init(elev=0., azim=-0)
# plt.show()
# optFrame.frameFE.plotStructure('Linear',plotDeformed = True,TrueScale=True,
#                 fig=plt.figure(10,figsize=(10,10)),thicknessPlot=False,nodeAnnotate=False)
# plt.show()

# print(aaa)
print("Optimization design variable size:", x_init.shape)
volume_max = (optFrame.frameFE.eleLength.sum().detach()*8.0)
if params is not None and params['Name']=='O':# O for octet, B for body centered cubic
    volume_max = torch.tensor([106429.890], dtype=torch.float64) # keep max volume same as of BCC for fair comparison
if int(exampleCase[0]) == 3:  
    vf = 0.5
elif int(exampleCase[0]) == 4 or int(exampleCase[0]) == 5:  
    vf = 0.3
elif int(exampleCase[0]) == 2:
    vf = 0.625
elif int(exampleCase[0]) == 1:
    vf = 0.4167 # 0.4167 or 0.8
    vf = 0.8

    
objType = 'SE' # SE or MSEbySE
if int(exampleCase[0]) < 6:
    conType={
            'Volume': vf*volume_max,
            # 'Manufacturability': 0.2,
        }
    
if int(exampleCase[0]) == 6:
    objType = 'MSEbySE' # for compliant mechanism
    conType = None
    
optimizationSetup = {
    'objective': objType,
    'constraints': conType
}

print(optimizationSetup)
opt_options={'maxiter':50,'disp':False,'move_limit':0.5,'maxfun':500,'kkttol':1e-5,
            'miniter':10}
fun_best = np.inf
n_start = 15
start_time = time.perf_counter()
for i in range(n_start):
    print(f"Starting optimization run {i+1} of {n_start} ")
    x_init = rng.uniform(size = x_init.shape)
    # x_init = np.array([1,1.,1,1]).reshape(-1,1)
    xopt,fun = optFrame.optimizerRun(optimizationSetup,x_init,algorithm='MMA',options=opt_options,fileSaveLoc=None)

    if fun < fun_best:
        fun_best = fun.copy()
        xopt_best = xopt.copy()
        print(f"Best objective: {fun_best} ")
total_time = time.perf_counter() - start_time
avg_time_per_run = total_time / (i+1)

In [ ]:
# xopt_best[-1] = 0
print("Problem:", exampleCase)
print("Symmetry type:", symType)
print("########## Optimization Results #########################")
J,dJ = optFrame.objectiveCall(xopt_best)
c,dc = optFrame.conCall(xopt_best) # type: ignore
print(f"Optimized objective J: {J.item():.2f}")
print(f"Optimized constraints c: {c.item():.2f}")
print("#########################################################")
print("Total optimization time (s):", total_time)
print("Average time per optimization run (s):", avg_time_per_run)
print("Number of FEA elements:", optFrame.frameFE.connectivity.shape[0])
print("Number of FEA dofs:", optFrame.frameFE.u.shape)
print("Number of design variable rows:", len(optFrame.varSetup['symArray']))
print(optimizationSetup)
print("#########################################################")
print("Optimal design variables:", xopt_best.reshape(-1))


In [ ]:
# # create a grid of parameters in xopt_best ranging from 0 to 1 and calculate obj, cons and find best 
# n= 10
# x1 = torch.linspace(0,1,n)
# x2 = torch.linspace(0,1,n)
# x3 = torch.linspace(0,1,n)
# x4 = torch.linspace(0,1,n)

# X1, X2, X3, X4 = torch.meshgrid(x1, x2, x3, x4, indexing='ij')
# X_grid = torch.stack((X1.reshape(-1),X2.reshape(-1),X3.reshape(-1),X4.reshape(-1)),dim=1)
# J_grid = []
# c_grid = []
# for i in range(X_grid.shape[0]):
#     J_temp, _ = optFrame.objectiveCall(X_grid[i,:].reshape((-1,1)))
#     c_temp, _ = optFrame.conCall(X_grid[i,:].reshape((-1,1))) # type: ignore
#     J_grid.append(J_temp.item())
#     c_grid.append(c_temp.item())
# J_grid = torch.tensor(J_grid).reshape(X_grid.shape[0])
# c_grid = torch.tensor(c_grid).reshape(X_grid.shape[0])

In [ ]:
# # find the c==0 such that minimum J
# mask = (c_grid <= 1e-3) & (c_grid >= -1e-3)
# J_feasible = J_grid.clone()
# J_feasible[~mask] = float('inf')
# min_J = torch.min(J_feasible)
# min_index_flat = torch.argmin(J_feasible)
# min_index_unraveled = torch.unravel_index(min_index_flat, J_feasible.shape)
# corresponding_c = c_grid[min_index_unraveled]
# print(f"Minimum J in grid: {min_J.item():.2f} with corresponding c: {corresponding_c.item():.2f}")
# # and min X_grid value
# xopt_best2 = X_grid[min_index_flat]
# print(f"Optimal design variables at min J: {xopt_best2.numpy()}")

In [ ]:

# optFrame.frameFE.plotStructure('Linear',plotDeformed = True,TrueScale=True,
#                 fig=plt.figure(10,figsize=(10,10)),thicknessPlot=False,nodeAnnotate=False)
# optFrame.frameFE.ax.view_init(elev=0., azim=-90)
# optFrame.frameFE.ax.grid(False)

# plt.show()
# optFrame.frameFE.plotStructure('Linear',plotDeformed = True,TrueScale=True,
#                 fig=plt.figure(10,figsize=(10,10)),thicknessPlot=False,nodeAnnotate=False)
# optFrame.frameFE.ax.view_init(elev=0., azim=-0)
# plt.show()
# axFD = optFrame.frameFE.plotStructure('Linear',plotDeformed = True,TrueScale=True,
#                 fig=plt.figure(10,figsize=(20,20)),thicknessPlot=False,nodeAnnotate=False)
# axFD.grid(False)
# plt.show()

if Section == 'circle':
    print("Ending circle optimization\n")
    print(aaa)
print(volume_max)


In [ ]:
# # For visualizing example 1 and 2 only random shape
# rng = np.random.default_rng(18)# 18 for example 1, and 2 for example 2
# xopt_best = rng.uniform(size=xopt_best.shape).reshape((-1,4))
# xopt_best[:,2] = 1.0;xopt_best[:,3] = 0.0;
# xopt_best = xopt_best.reshape((-1,1))


In [ ]:
xopt_best_final = torch.tensor(xopt_best.reshape((x0.shape)));
xopt_best_final_denorm =  optFrame.apply_var_denormalize(xopt_best_final);
CP_xy, theta_all = shapePredict(torch.tensor(xopt_best_final_denorm))
prop = propPredict(torch.tensor(xopt_best_final_denorm))
prop = torch.cat((torch.ones_like(prop[:,0]).reshape((-1,1)),prop),dim=1)

    
xy_final = splineGeometry.rotate_cross_section(CP_xy[0,:].detach().numpy(),theta=theta_all[0].item())
XY_all = np.zeros((CP_xy.shape[0],xy_final.shape[0],xy_final.shape[1]))
XY_all[0,:,:] = xy_final.copy()
for i in range(1,CP_xy.shape[0]):
    xy_final = splineGeometry.rotate_cross_section(CP_xy[i,:].detach().numpy(),theta=theta_all[i].item())
    XY_all[i,:,:] = xy_final.copy()

prop_np = prop.detach().numpy()
prop_rounded = np.max(prop_np,axis=0) * np.ceil((prop_np/np.max(prop_np,axis=0))*100)/100
# find unique shapes using properties
_, unique_indices = np.unique(
    prop_rounded,
    axis=0,
    return_index=True
)

print(f"{len(unique_indices)} of unique cross-sectional shapes in optimized design out of {prop_np.shape[0]}")
print("##############")
nn = 5000
if len(unique_indices) < nn:
    unique_indices = np.sort(unique_indices)
    # subplot grid (based on number of unique shapes)
    n = min(49,len(unique_indices))
    n1 = int(np.sqrt(n))
    n2 = int(np.ceil(n / n1))
    fig_opt, axes_opt = plt.subplots(n1, n2, figsize=(n2*5, n1*5))
    try:
        axes_in_opt = axes_opt.flatten()
    except:
        axes_in_opt = [axes_opt]
    good_area_idx_mask = np.where(prop_rounded[unique_indices,2] > 0.0003)[0][:n]
    
    # plot unique shapes only
    for k, i in enumerate(unique_indices[good_area_idx_mask]):
        splineGeometry.plot_cross_section(
            CP_xy[i,:].detach().numpy(),
            prop[i,:].detach().numpy(),
            theta=theta_all[i].item(),
            ax=axes_in_opt[k],
            addLegend=False, addFill=True
        )
        
srcLoc = 'PaperImg/'
if int(exampleCase[0]) == 1:
    ExmplLoc = 'SingleBeam'
elif int(exampleCase[0]) == 2:
    ExmplLoc = 'ThreeBeam'
elif int(exampleCase[0]) == 3:
    ExmplLoc = 'Lattice'
    if symType == 'OneValue':
        ExmplLoc = 'Lattice/Lattice1'
    if symType == 'Midplane': # 'Unitcell', 'OneValue', 'Independent', 'Midplane'
        ExmplLoc = 'Lattice/Lattice3'
    if symType == 'Unitcell':
        ExmplLoc = 'Lattice/Lattice36'
    if symType == 'Independent':
        ExmplLoc = 'Lattice/Lattice276'
elif int(exampleCase[0]) == 4:
    ExmplLoc = 'Cantilever'
elif int(exampleCase[0]) == 5:
    ExmplLoc = 'MBB'
elif int(exampleCase[0]) == 6:
    ExmplLoc = 'Inverter'  #  compliant inverter
    
ExmplLoc = ExmplLoc + '/'+exampleCase[1::] # type: ignore
if int(exampleCase[0])>2:
    if conType is not None and 'Volume' in conType:
        ExmplLoc = ExmplLoc + '_vf'+str(int(10*vf))+'by10' # type: ignore
    else:
        ExmplLoc = ExmplLoc + '_unconstrained' # type: ignore
if len(unique_indices) < nn:
#     # save fig
    if add_thickness:
        fig_opt.savefig(srcLoc + ExmplLoc + '_VAE_v_192.png', dpi=300)
    else:
        fig_opt.savefig(srcLoc + ExmplLoc + '_VAE.png', dpi=300)


In [ ]:
if add_thickness:
    print("End code, no stl generation for thickness case") # 
    print(aaa)

In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components


def deleteElements(nodeXYZ, connectivity, Areas, threshold_A=0.1):
    """
    Deletes elements with small area AND removes hanging (disconnected) elements.

    Returns:
        new_nodeXYZ
        new_connectivity
        deleted_element_indices (w.r.t. original connectivity)
    """

    n_elems = connectivity.shape[0]

    # Track original element indices
    elem_ids = np.arange(n_elems)

    # -------------------------------
    # 1. Area-based deletion
    # -------------------------------
    keep_area = Areas > threshold_A

    deleted_area = elem_ids[~keep_area]

    connectivity = connectivity[keep_area]
    elem_ids = elem_ids[keep_area]

    # -------------------------------
    # 2. Remove unused nodes
    # -------------------------------
    used_nodes = np.unique(connectivity)
    node_map = {old: new for new, old in enumerate(used_nodes)}

    nodeXYZ = nodeXYZ[used_nodes]
    connectivity = np.vectorize(node_map.get)(connectivity)

    # -------------------------------
    # 3. Remove hanging components
    # -------------------------------
    n_nodes = nodeXYZ.shape[0]

    rows = np.hstack([connectivity[:, 0], connectivity[:, 1]])
    cols = np.hstack([connectivity[:, 1], connectivity[:, 0]])
    data = np.ones(len(rows))

    graph = csr_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes))

    n_comp, labels = connected_components(graph, directed=False)

    # Find largest connected component
    comp_sizes = np.bincount(labels)
    main_comp = np.argmax(comp_sizes)

    # Elements fully inside main component
    keep_hanging = np.all(labels[connectivity] == main_comp, axis=1)

    deleted_hanging = elem_ids[~keep_hanging]

    connectivity = connectivity[keep_hanging]
    elem_ids = elem_ids[keep_hanging]

    # -------------------------------
    # 4. Final node cleanup
    # -------------------------------
    used_nodes = np.unique(connectivity)
    node_map = {old: new for new, old in enumerate(used_nodes)}

    new_nodeXYZ = nodeXYZ[used_nodes]
    new_connectivity = np.vectorize(node_map.get)(connectivity)

    # -------------------------------
    # 5. Combine deleted indices
    # -------------------------------
    deleted_element_indices = np.sort(
        np.concatenate([deleted_area, deleted_hanging])
    )

    return new_nodeXYZ, new_connectivity, deleted_element_indices


In [ ]:
from beamBuild  import BeamStructureSTL
import trimesh.creation as creation
import trimesh

Area_opt = prop[:,2].detach().numpy()

beam_struct = BeamStructureSTL()
beam_struct.verbose = False
nodes = torch.tensor(optFrame.frameFE.nodeXYbase).detach().numpy()
beams = torch.tensor(optFrame.frameFE.connectivityBase).detach().numpy()
orientations = torch.tensor(optFrame.frameFE.v_base).detach().numpy()

print("Before:",nodes.shape, beams.shape)
th_A = 0.3
nodes, beams, deleted_elem_indices = deleteElements(
    nodes, 
    beams, 
    Area_opt, # Area
    threshold_A=th_A
)
orientations = np.delete(orientations, deleted_elem_indices, axis=0)

from scipy.ndimage import gaussian_filter1d
XY_alln = XY_all.copy()
for i in range(100,200,5):
    try:
        # i is the number of points defining each cross-section
        indicesToSample = np.linspace(0, XY_all.shape[1] - 2, i, dtype=int) # type: ignore
                    
        XY_sampled = (
            torch.tensor(XY_alln)[:, indicesToSample, :]
            .reshape((XY_all.shape[0], -1))# type: ignore
        ) 
        
        sections = XY_sampled.detach().numpy()
        sections = np.delete(sections, deleted_elem_indices, axis=0)
        prop_rounded_deleted = np.delete(prop_rounded, deleted_elem_indices, axis=0)
        # find unique sections and assign colors for each
        unique_sections = np.unique(prop_rounded_deleted, axis=0, return_inverse=True)
        section_indices = unique_sections[1]
        num_unique_sections = len(unique_sections[0])
        colors = plt.cm.rainbow(np.linspace(0, 1, num_unique_sections)) # type: ignore
        sections_color = colors[section_indices]
        
        filename = 'STL/'+ExmplLoc+'_optimized_beam_structure.stl'
        # mesh,mesh_colored = beam_struct.build_structure(
        #     nodes,
        #     beams,
        #     sections,
        #     orientations,
        #     sections_color,
        #     extension=-0.025,
        #     stl_filename=filename,
        #     visualize=False
        # )
        break
    except Exception as e:
        print("Error in building beam structure STL:", e)
        XY_alln = gaussian_filter1d(XY_all, sigma=i, axis=1)
        continue
    
print("After:",nodes.shape, beams.shape)


In [ ]:
# mesh = trimesh.load(filename)
axes = creation.axis(origin_size=.50, axis_length=20)

COLORS = {
    "light_blue":       [173, 216, 230, 255],
    "steel_gray":       [180, 180, 180, 255],   
    "soft_teal":        [120, 190, 190, 255],
    "light_orange":     [255, 200, 120, 255],
    "muted_green":      [150, 200, 150, 255],
    "dark_blue":        [60, 90, 160, 255],
    "transparent_gray": [180, 180, 180, 180],
    "transparent_blue": [0, 0, 255, 250],
}
mesh.visual.face_colors = COLORS["transparent_blue"]
scene = trimesh.Scene([mesh, axes]) # colorless but has joint combined
scene2 = trimesh.Scene([mesh_colored,axes])
isometric = True
if isometric:
    if int(exampleCase[0]) == 1:
        camera_transform = np.array([
            [ 0.4681, -0.2504,  0.8475,  51.2003],
            [ 0.8823,  0.0796, -0.4639, -17.2323],
            [ 0.0488,  0.9648,  0.2582,  14.9987],
            [ 0.0000,  0.0000,  0.0000,   1.0000]
        ])
    
    elif int(exampleCase[0]) == 2:
        camera_transform = np.array([
            [-0.8528,  0.1887, -0.4869, -15.3326],
            [-0.5221, -0.2931,  0.8010,  63.8915],
            [ 0.0085,  0.9372,  0.3485,  40.8055],
            [ 0.0000,  0.0000,  0.0000,   1.0000]
        ])

    elif int(exampleCase[0]) == 3:
        camera_transform = np.array([
            [ 0.7085,  0.2486, -0.6604, -38.0000],
            [-0.7056,  0.2384, -0.6673, -38.0000],
            [-0.0085,  0.9388,  0.3443,  38.0000],
            [ 0.0000,  0.0000,  0.0000,   1.0000]
        ])
    
    elif int(exampleCase[0]) == 4:
        camera_transform = np.array([
            [ 0.6613, -0.3133,  0.6816, 167.3898],
            [ 0.7501,  0.2641, -0.6063, -59.8340],
            [ 0.0099,  0.9122,  0.4096,  65.4784],
            [ 0.0000,  0.0000,  0.0000,   1.0000]
        ])
    elif int(exampleCase[0]) == 5:
        camera_transform = np.array([
            [ 0.6613,   -0.3133,    0.6816,   311090.62843 ],
            [ 0.7501,    0.2641,   -0.6063,   -94.6215488 ],
            [ 0.0099,    0.9122,    0.4096,    80.8128588 ],
            [ 0.0,       0.0,       0.0,        1.0 ]
        ])  
    elif int(exampleCase[0]) == 6:
        camera_transform = np.array([[  0.8529,  -0.3106,   0.4197,  92.8453],
            [  0.5195,   0.5838,  -0.6239, -73.8163],
            [ -0.0513,   0.7502,   0.6592, 107.3633],
            [  0.0000,   0.0000,   0.0000,   1.0000]])  
            
        camera_transform = np.array([[   1.0000,   -0.0019,   -0.0003,   37.5072],
            [  -0.0003,   -0.0098,   -0.9999, -119.3732],
            [   0.0017,    1.0000,   -0.0098,   20.4179],
            [   0.0000,    0.0000,    0.0000,    1.0000]])
    scene.camera_transform = camera_transform
    scene2.camera_transform = camera_transform


In [ ]:
scene.show(jupyter=True)

In [ ]:
scene2.show(jupyter=True)

In [ ]:
res1,res2 = 2000,1500
png = scene.save_image(resolution=(res1, res2),
    visible=True
)
open(srcLoc+ ExmplLoc + "_Opt_singleColor.png", "wb").write(png)

png = scene2.save_image(resolution=(res1, res2),
    visible=True
)
open(srcLoc+ ExmplLoc + "_Opt_multiColor.png", "wb").write(png)

In [ ]:
# plot_x_points_inlatentspace(xopt_best_final[:,0:2],figLatent,axsLatent)

In [ ]:

# exampleInfo, nodeXY, connectivity, bc, radiiNodIndex,radiiElemIndex = getExample(exampleCase,params) ##3D beam model

# numUnitLatElem = len(beams[:,0])
# numUnitLatNode = len(nodes[:,0])
# meshSetting = {'nodeXY':nodes,'connectivity':beams,'numElemsPerBeam':numElemsPerBeam,'elemSize':elemSize,
#                'ElemType':'TS','radiiElemIndex':radiiElemIndex,'radiiNodIndex':radiiNodIndex,
#                'numUnitLatElem':numUnitLatElem,'numUnitLatNode':numUnitLatNode}# and TS - Timoshenko and EB - Euler-Bernoulli
# AnalysisSettings = {'Type':'Linear','solver':solverName,'matrixType':matrixType,'Section':Section}# mode {analysis or optimization} true uses torch sp solve
# frameFE = FrameFE(meshSetting,bc,AnalysisSettings)
# optFrame.frameFE = frameFE
# xopt_best_updated = xopt_best.reshape((-1,4))
# xopt_best_updated = np.delete(xopt_best_updated,deleted_elem_indices,axis=0)
# optFrame.varSetup['symArray'] = None # each crosssection independent now
# _,_ = optFrame.objectiveSE(xopt_best_updated.reshape((-1,4)))
# axe = optFrame.frameFE.plotStructure('Linear',plotDeformed = True,TrueScale=True,
#         fig=plt.figure(10,figsize=(20,20)),thicknessPlot=False,nodeAnnotate=False)
# axe.grid(False)
# plt.savefig(srcLoc +ExmplLoc + "_Opt_Structure_Deformed.png", dpi=300)
# optFrame.frameFE.ax.view_init(elev=0., azim=-90)
# plt.savefig(srcLoc +ExmplLoc + "_Opt_Structure_Deformed2.png", dpi=300)
# plt.show()